# AdaBoost Discreto

In [ ]:
from BUSBRA_Medical import AdaDataModule
from classifiers import ClassificationModel
import pytorch_lightning as pl
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

## Setando seeds para reprodutibilidade

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    # Set the seed for Python's built-in random library
    random.seed(seed)
    
    # Set the seed for NumPy
    np.random.seed(seed)
    
    # Set the seed for PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    
    # For performance reasons, this setting should only be enabled for true reproducibility.
    # It can lead to slower training in some cases.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set the seed for PyTorch Lightning
    pl.seed_everything(seed, workers=True)

# Call the function to set seeds
set_seed(42)

## Carregando CSV do dataset

In [ ]:
import pandas as pd

merged_df = df = pd.read_csv('../busbra_medical_error.csv')
merged_df.head()

# Montagem do ensemble

In [ ]:
def set_trainer():      
    max_epochs = 2
    patience = 20
    
    early_stop_callback = EarlyStopping(
                            monitor="val_loss", 
                            patience=patience, 
                            verbose=False, 
                            mode="min"
                            )

    trainer = pl.Trainer(
                accelerator="gpu", 
                devices=1, 
                precision='16-mixed',
                max_epochs=max_epochs,
                callbacks = [early_stop_callback, 
                             # checkpoint_callback
                            ],
                accumulate_grad_batches=2,
                num_sanity_val_steps=0
            )
    return trainer

In [ ]:
# treina modelo unico
from sklearn.model_selection import KFold
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from matplotlib import pyplot as plt
from datetime import datetime
from adasampling import AdaSampler

def train_single_model(dm, model_name='resnet18',fold=[1,2,3,4,5]):
    moment = datetime.now().strftime("%d-%m-%y-%H-%M-%S")
    
    hp = [0.0001, 0.9, 0.0001, 10]
    
    Kfold = 5
    
    for i in range(Kfold):
        if i+1 in fold:
            print(f'####### Fold {i+1} #######')  
        
            count_classes = [len(train_df[train_df['Pathology'] == 'benign']), len(train_df[train_df['Pathology'] == 'malignant'])]
            class_weights = 1/torch.tensor(count_classes)
            class_weights = class_weights/torch.mean(class_weights)
                
            model_hparams={"in_channels": 3, "num_classes": 2, "act_fn_name": "relu"}
            optimizer_name="SGD"
            optimizer_hparams={"lr": hp[0], "momentum": hp[1], "weight_decay": hp[2]}
            max_epochs = 100
            patience = 20
            
            early_stop_callback = EarlyStopping(
                                    monitor="val_loss", 
                                    patience=patience, 
                                    verbose=False, 
                                    mode="min"
                                    )
        
            checkpoint_callback = ModelCheckpoint(
                 dirpath='ensemble_weights\\swint',
                 filename=f'best-BUSBRA-{model_name}-real-fold-{i+1}-{moment}',
                 monitor='val_loss',
                 mode="min",
                )            
            trainer = pl.Trainer(
                accelerator="gpu", 
                devices=1, 
                precision='16-mixed',
                max_epochs=max_epochs,
                callbacks = [early_stop_callback, 
                            ],
                accumulate_grad_batches=2,
                num_sanity_val_steps=0
            )
        
            model = ClassificationModel(model_name, model_hparams, optimizer_name, 
                                        optimizer_hparams, 
                                        loss_weight=class_weights
                                       ) 
            
            trainer.fit(
                model=model, 
                datamodule=dm
                )
            
            model.sweights, model.alpha = update_adaboost(dm, model)
            
            val_metrics = trainer.validate(model=model, datamodule=dm,
                                          verbose=False)
    return model

In [ ]:
from  torch.cuda.amp import autocast
import pytorch_lightning as pl
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, mean_absolute_error
import torch.nn as nn


class MyAdaBoostEnsemble(pl.LightningModule):
    def __init__(self,
                models, loss_weight=None
                ):
        super(MyAdaBoostEnsemble, self).__init__()
        self.models = models
        self.cuda0 = torch.device('cuda:0')
        self.training_step_outputs = []
        self.validation_step_outputs = []
        self.test_step_outputs = []
        self.weights = [5,4,3,2,1]

        for i in range(len(self.models)):
            self.models[i].freeze()
            self.models[i].to(self.cuda0)
        
        self.loss_module = nn.CrossEntropyLoss(weight=loss_weight)
        
    def forward(self, x):
        x0 = torch.nn.functional.softmax(self.models[0](x) * self.models[0].alpha)  # logits * alpha
        for i in range(1,len(self.models)):
            xi = torch.nn.functional.softmax(self.models[i](x) * self.models[i].alpha)  # logits * alpha
            x0 = x0 + xi  # soma dos logits ponderados
        x0 = x0/len(self.models)
        return x0
    
    def shared_step(self, batch, stage):
        imgs, labels = batch
        
        with autocast():
            preds = self.forward(imgs)
            loss = self.loss_module(preds, labels)
                
        outputs = {
            "loss": loss,
            "labels": labels,
            "preds": preds
        }
        
        if stage == 'train':
            self.training_step_outputs.append(outputs)

        return outputs
    
    def shared_epoch_end(self, outputs, stage):
        labels = torch.cat([x["labels"] for x in outputs]).cpu()
        probs = torch.cat([x["preds"] for x in outputs]).cpu()  # logits softmax já aplicados no forward
        preds = (probs[:, 1] > 0.457).long()
        loss = torch.cat([x["loss"].reshape(1) for x in outputs]).cpu()
        
        acc = (preds == labels).float().mean()
        loss = loss.mean()
        
        if stage == 'test':
        
            cm = confusion_matrix(labels, preds)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['benign', 'malignant'])
            disp.plot(cmap='Blues')

            plt.show()
        
        metrics = {
             f"{stage}_acc": acc,
             f"{stage}_f1_score": f1_score(labels, preds),
             f"{stage}_sens": recall_score(labels, preds),
             f"{stage}_spec": recall_score(labels, preds, pos_label=0),
             f"{stage}_loss": loss            
         }
        
        self.log_dict(metrics, prog_bar=True)
        
    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def on_train_epoch_end(self):
        outputs = self.training_step_outputs.copy()
        self.training_step_outputs.clear()
        return self.shared_epoch_end(outputs,"train")

    def validation_step(self, batch, batch_idx):
        self.validation_step_outputs.append(self.shared_step(batch, "val"))
        return self.shared_step(batch, "val")
    
    def on_validation_epoch_end(self):
         return self.shared_epoch_end(self.validation_step_outputs, 'val')

    def test_step(self, batch, batch_idx):
        self.test_step_outputs.append(self.shared_step(batch, "test"))
        return self.shared_step(batch, "test")  

    def on_test_epoch_end(self):
        return self.shared_epoch_end(self.test_step_outputs, 'test')

    def configure_optimizers(self):
        return optim.SGD(self.parameters(), **self.hparams.optimizer_hparams)

In [ ]:
def update_adaboost(dm, new_model):
    err = np.sum(new_model.is_wrong.numpy()*dm.sweights)/dm.sweights.sum()
    err = np.clip(err, 1e-10, 1 - 1e-10)
    # adicionando divisao por 2 para diminuir o peso
    new_alpha = 1/2*np.log((1 - err)/err) + np.log(new_model.hparams['model_hparams']['num_classes']-1)
    new_sweights = dm.sweights * np.exp(new_alpha*new_model.is_wrong.numpy())
    new_sweights = new_sweights/new_sweights.sum()

    return new_sweights, new_alpha

In [ ]:
def evaluate_ensemble_step(dm, ensemble_list, new_models):
    print('chamou evaluate ensemble step')
    trainer = set_trainer()
    ensemble_f1 = 0
    for model in new_models:
        temp_ensemble_list = ensemble_list.copy()
        
        temp_ensemble_list.append(model)
        temp_ensemble = MyAdaBoostEnsemble(temp_ensemble_list)
        val_metrics = trainer.validate(model=temp_ensemble, datamodule=dm,
                                      verbose=False)
        if val_metrics[0]['val_f1_score'] > ensemble_f1:
            ensemble_f1 = val_metrics[0]['val_f1_score']
            new_model = model
            best_val_metrics = val_metrics
    dm.sweights = new_model.sweights
    ensemble_list.append(new_model)
    return ensemble_list, best_val_metrics

In [ ]:
num_models = 9
ensemble_list = []
ensemble_history = {'val_f1_score': [],
                    'val_loss':[],
                    'val_f1_score': [],
                    'val_loss': []
                        }

model_bag = ['resnet18', 'convnext', 'swint', 
            ]

# change for fold 1, 2, 3, 4 and 5
fold = [1]
column = 'valid_' + str(fold[0])
train_df = merged_df[merged_df[column] == 1]
val_df = merged_df[merged_df[column] == 0]
test_df = merged_df[merged_df['kFold'] == fold[0]]
train_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)  
test_df.reset_index(drop=False)
dm = AdaDataModule(train=train_df, val=val_df, 
                      test=test_df)

# setar adaboost, que calcula ada_alpha
for i in range(num_models):
    print(f' {i+1} iteração do ensemble')
    # lista de modelos treinados que vão ser validados para se encaixarem no 
    # ensemble
    candidate_models = []
    for model_name in model_bag:        
        # treina um modelo de cada arquitetura
        candidate_models.append(train_single_model(dm=dm,
                                                   model_name=model_name,
                                                   fold=fold
                                                   ))
                                            

        # tem que atualizar o sweights aqui, com info do modelo fraco
    # avalia qual dos modelos treinados deve ser acoplado ao ensemble
    ensemble_list,ensemble_metrics = evaluate_ensemble_step(dm, ensemble_list, 
                                      candidate_models)
    ensemble_history['val_f1_score'].append(ensemble_metrics[0]['val_f1_score'])
    ensemble_history['val_loss'].append(ensemble_metrics[0]['val_loss'])

In [ ]:
# val
train_df = merged_df[merged_df[column] == 1]
val_df = merged_df[merged_df[column] == 0]
test_df = merged_df[merged_df['kFold'] == fold[0]]
train_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)  
test_df.reset_index(drop=False,inplace=True)  
dm = AdaDataModule(train=train_df, val=val_df, 
                      test=test_df)
trainer = set_trainer()
ensemble_model = MyAdaBoostEnsemble(ensemble_list)
results = trainer.validate(model=ensemble_model, datamodule=dm)

In [ ]:
# teste
train_df = merged_df[merged_df[column] == 1]
val_df = merged_df[merged_df[column] == 0]
test_df = merged_df[merged_df['kFold'] == fold[0]]
train_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)  
test_df.reset_index(drop=False,inplace=True)  
dm = AdaDataModule(train=train_df, val=val_df, 
                      test=test_df)
trainer = set_trainer()
ensemble_model = MyAdaBoostEnsemble(ensemble_list)
results = trainer.test(model=ensemble_model, datamodule=dm)

In [ ]:
# teste
pred2 = []
pred3 = []
pred4 = []
pred5 = []

y2 = []
y3 = []
y4 = []
y5 = []

model_tp_count = 0
model_tn_count = 0
doctor_tp_count = 0
doctor_tn_count = 0

model2_tp_count, model3_tp_count, model4_tp_count, model5_tp_count = 0, 0, 0, 0
model2_tn_count, model3_tn_count, model4_tn_count, model5_tn_count = 0, 0, 0, 0
doctor2_tp_count, doctor3_tp_count, doctor4_tp_count, doctor5_tp_count = 0, 0, 0, 0
doctor2_tn_count, doctor3_tn_count, doctor4_tn_count, doctor5_tn_count = 0, 0, 0, 0


train_df = merged_df[merged_df[column] == 1]
val_df = merged_df[merged_df[column] == 0]
test_df = merged_df[merged_df['kFold'] == fold[0]]
train_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)
val_df.reset_index(drop=False,inplace=True)  
test_df.reset_index(drop=False,inplace=True)  
dm = AdaDataModule(train=train_df, val=val_df, 
                      test=test_df)
trainer = set_trainer()
ensemble_model = MyAdaBoostEnsemble(ensemble_list[:4])

preds = trainer.predict(model=ensemble_model, datamodule=dm)
predsm = torch.cat([x for x in preds]).argmax(dim=-1)

for pred, y, birad, name, medical_error in zip(predsm, test_df['Pathology'].values,test_df['BIRADS'].values, test_df['ID'].values, test_df['Medical error'].values):
    y = 1 if y=='malignant' else 0    

    if (birad==2 or birad==3) and y==0:
        doctor_tn_count += 1
        if pred==y:
            model_tn_count += 1   
    elif (birad==4 or birad==5) and y==1:
        doctor_tp_count += 1
        if pred==y:
            model_tp_count += 1
    
    if birad == 2:
        pred2.append(int(pred))
        y2.append(y)
    elif birad == 3:
        pred3.append(int(pred))
        y3.append(y)
    elif birad == 4:
        pred4.append(int(pred))
        y4.append(y)
    elif birad == 5:
        pred5.append(int(pred))
        y5.append(y)
fold_cm = []
true_list = []

true_list.append([model_tp_count, doctor_tp_count, model_tn_count, doctor_tn_count])

birads_dict = {'2':confusion_matrix(y2, pred2, labels=[0, 1]),
                   '3':confusion_matrix(y3, pred3, labels=[0, 1]),
                   '4':confusion_matrix(y4, pred4, labels=[0, 1]),
                   '5':confusion_matrix(y5, pred5, labels=[0, 1])
                  }
fold_cm.append(birads_dict)

ms_fn = 0
ms_fp = 0
for i in range(len(fold_cm)):
    ms_fn += fold_cm[i]['2'][1][1] + fold_cm[i]['3'][1][1]
    ms_fp += fold_cm[i]['4'][0][0] + fold_cm[i]['5'][0][0]

m_tp = 0
d_tp = 0
m_tn = 0
d_tn = 0
for i in range(len(true_list)):
    m_tp += true_list[i][0]
    d_tp += true_list[i][1]
    m_tn += true_list[i][2]
    d_tn += true_list[i][3]

# print para o LaTeX
results_latex = f"Ensemble Vanilla & {round(100*ms_fn / 284, 2)}%({ms_fn}/41) & {round(100*ms_fp / 284, 2)}%({ms_fp}/284) & {round(100*m_tn / d_tn, 2)}%({m_tn}/{d_tn}) & {round(100*m_tp / d_tp, 2)}\\%({m_tp}/{d_tp})"
print('Model            | M/S-FN       | M/S-FP | M/S-TN | M/S-TP')
print(results_latex)